# 00 Load Config

Load `cfg_metadata_source.csv` from Lakehouse Files into `governance.cfg_metadata_source`.

In [ ]:
from pyspark.sql import functions as F

CATALOG_SCHEMA = "governance"
CONFIG_PATH = "Files/data_dictionary/config/cfg_metadata_source.csv"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG_SCHEMA}")

In [ ]:
cfg_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(CONFIG_PATH)
)

cfg = (
    cfg_raw
    .withColumn("source_id", F.trim(F.col("source_id")))
    .withColumn("layer", F.trim(F.col("layer")))
    .withColumn("workspace_name", F.trim(F.col("workspace_name")))
    .withColumn("lakehouse_name", F.trim(F.col("lakehouse_name")))
    .withColumn("schema_name", F.trim(F.col("schema_name")))
    .withColumn("domain", F.trim(F.col("domain")))
    .withColumn("is_active", F.lower(F.trim(F.col("is_active"))).isin("true", "1", "yes", "y"))
    .withColumn("include_table_pattern", F.coalesce(F.col("include_table_pattern"), F.lit(".*")))
    .withColumn("exclude_table_pattern", F.coalesce(F.col("exclude_table_pattern"), F.lit("")))
    .withColumn("owner_team", F.coalesce(F.col("owner_team"), F.lit("Data Engineering")))
    .withColumn("scan_row_count", F.lower(F.trim(F.col("scan_row_count"))).isin("true", "1", "yes", "y"))
    .withColumn("created_at", F.current_timestamp())
    .withColumn("updated_at", F.current_timestamp())
)

cfg.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG_SCHEMA}.cfg_metadata_source")

display(spark.table(f"{CATALOG_SCHEMA}.cfg_metadata_source"))